# 下载 Zhuang 数据并使用真实 CCF 坐标进行 3D 可视化

默认下载 MIMYR 作者的 **Zhuang-ABCA-2**：66 张冠状切片，压缩包 376,293,233 字节，解压约 5.9 GB。
建议预留至少 7 GB 磁盘空间。也支持 `DATASET = 3`（23 张矢状切片）。

**仅下载和浏览原始数据，不训练、不推理、不修改论文实验配置。** 下载支持 `.part` 断点续传，
完成后校验 SHA-256，解压时校验 ZIP CRC，并保留已有且内容一致的文件。

作者数据的真实 CCF 坐标在 `ccf-ABCA-2.csv` 中，需要按 `cell_label` 与 `.h5ad` 的 `obs_names` 关联。
这里保留长整数形式的完整字符串 ID，不使用行号关联，也不把原始 `spatial` 当成 CCF。
没有 CCF 的细胞会从可视化中排除，并在每张切片的审查表中计数。

来源：[作者数据仓库](https://huggingface.co/datasets/gkrieg13/mimyr-data)；
[Allen 官方数据说明](https://alleninstitute.github.io/abc_atlas_access/descriptions/Zhuang-ABCA-2.html)。
原始数据中的部分前后端切片无法配准，因此不能假定每张切片都有完整 CCF 覆盖。

3D 图使用真实的 CCF **x/y/z、单位 mm**，不交换或平移坐标；三个轴保持实际比例。
默认仅随机显示 120,000 个已配准细胞，以控制浏览器内存。抽样仅影响显示，不删减下载的数据。

In [ ]:
from pathlib import Path
import sys

PROJECT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'scripts/zhuang_ccf_3d.py').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('请从 MIMYR 或其 notebooks 目录打开本 notebook，并同步 scripts/zhuang_ccf_3d.py。')
sys.path.insert(0, str(PROJECT))
print('项目目录:', PROJECT)
print('Python 内核:', sys.executable)

## 1. 依赖

使用已有 Python 环境即可，不需要 conda、CUDA 或 PyTorch。
如果缺少依赖，将下一行的注释去掉再运行。新安装的依赖可能需要重启内核。

In [ ]:
# %pip install -r "$PROJECT/requirements.zhuang-visualization.txt"

In [ ]:
import json
import pandas as pd
from IPython.display import display
from scripts.zhuang_ccf_3d import (
    DATASETS, SOURCE, download_archive, extract_archive,
    load_ccf_cells, visualization_sample, write_3d_html,
)

DATASET = 2
DATA_DIR = PROJECT / 'data'
OUTPUT_DIR = PROJECT / 'results/zhuang_ccf_3d'
SLICES = None             # 全部切片；也可以设成 [31, 32, 33, 34, 35]
MAX_POINTS = 120_000      # 只限制浏览器显示点数，下载数据始终完整
SEED = 42

if DATASET not in DATASETS:
    raise ValueError('DATASET 支持 2 或 3。')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('来源:', SOURCE)
print('数据:', DATA_DIR)
print('预期压缩包字节数:', DATASETS[DATASET]['size'])
print('预期 SHA-256:', DATASETS[DATASET]['sha256'])

## 2. 下载并验证

这个单元会真正联网下载到本地 `MIMYR/data/`。若网络中断，保留 `.zip.part` 并重新运行即可继续。
已有完整 ZIP 会再次核对大小和 SHA-256，不重复下载。

In [ ]:
archive = download_archive(DATA_DIR, DATASET)
print('已验证:', archive)

## 3. 解压原始数据

不覆盖内容不同的已有文件。再次运行会核对已有文件的大小和 CRC；因此即使不重新解压，也可能需要一段读取时间。

```text
MIMYR/data/
├── Zhuang-ABCA-2.zip
├── ccf-ABCA-2.csv
└── Zhuang-ABCA-2/
    ├── sec_001.h5ad
    ├── sec_002.h5ad
    └── ...
```

In [ ]:
section_dir = extract_archive(archive, DATA_DIR, DATASET)
section_files = sorted(section_dir.glob('sec_*.h5ad'))
print('切片目录:', section_dir)
print('切片数量:', len(section_files))
print('CCF 表:', DATA_DIR / f'ccf-ABCA-{DATASET}.csv')

## 4. 按真实 cell ID 关联 CCF

以 `backed='r'` 方式读取每张切片，只取细胞标签，不将全部表达矩阵载入内存。
CCF 的 `x/y/z` 另存为 `x_ccf/y_ccf/z_ccf`，避免与实验原始坐标混淆。
审查表中的 `n_without_ccf` 是未匹配或无有效 CCF 坐标的细胞数。

In [ ]:
cells, audit = load_ccf_cells(DATA_DIR, DATASET, SLICES)
audit_path = OUTPUT_DIR / f'Zhuang-ABCA-{DATASET}_ccf3d.audit.csv'
audit.to_csv(audit_path, index=False)
display(audit)
print('输入细胞:', f"{audit.n_input.sum():,}")
print('已配准细胞:', f'{len(cells):,}')
print('未匹配/无有效 CCF:', f"{audit.n_without_ccf.sum():,}")
display(cells.head())

## 5. 生成离线交互式 3D 图

HTML 内置 Plotly，无需联网打开。可旋转、缩放、筛选单张切片或细胞类型，并切换按切片/细胞类型着色；
细胞类型沿用作者的 `class` 标签及原始颜色。悬停显示 cell ID、切片、类别和 CCF 坐标。

这是已配准细胞点云，没有额外添加或虚构脑表面网格。

In [ ]:
sample = visualization_sample(cells, MAX_POINTS, SEED)
HTML_PATH = OUTPUT_DIR / f'Zhuang-ABCA-{DATASET}_ccf3d.html'
write_3d_html(sample, HTML_PATH, DATASET, total_cells=len(cells))
summary = {
    'dataset': DATASET, 'source': SOURCE,
    'n_registered_cells': len(cells), 'n_visualized_cells': len(sample),
    'visualization_seed': SEED, 'coordinate_units': 'mm', 'slices': SLICES,
    'archive_sha256': DATASETS[DATASET]['sha256'],
}
HTML_PATH.with_suffix('.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('完整 CCF 细胞:', f'{len(cells):,}', '| 显示细胞:', f'{len(sample):,}')
print('在浏览器中打开此文件:', HTML_PATH.resolve())
print('元数据:', HTML_PATH.with_suffix('.json'))
print('各切片 CCF 覆盖率:', audit_path)

In [ ]:
# 本地运行时可打开浏览器。若 notebook 在服务器上运行，请先下载生成的 HTML 到本地。
import webbrowser
OPEN_BROWSER = False
if OPEN_BROWSER:
    webbrowser.open(HTML_PATH.resolve().as_uri())

# 如需内嵌到 notebook，先信任 notebook，再按需取消以下注释。
# 注意：保存内嵌输出会明显增大 notebook 体积。
# from IPython.display import HTML
# display(HTML(filename=str(HTML_PATH)))

## 独立脚本用法

在项目根目录终端执行：

```bash
# 下载、校验、解压并绘制全部已配准切片
python scripts/zhuang_ccf_3d.py --download --dataset 2

# 已下载后，只绘制中部五张切片
python scripts/zhuang_ccf_3d.py --dataset 2 --slices 31 32 33 34 35 --max-points 60000
```

默认输出 `results/zhuang_ccf_3d/Zhuang-ABCA-2_ccf3d.html`。审查表和 JSON 会写在同一目录。
后一次使用不同切片/点数会更新同名图；需要同时保留两张图时，使用 `--output` 指定不同 HTML 名称。

下载数据仍是作者发布的完整数据；这里的显示抽样不是训练或评估抽样，也没有改动任何模型权重和实验脚本。